# Amazon ESCI公式Task 1作法によるOpenSearch評価（日本語test）

Amazon ESCIの公式Task 1（Query-Product Ranking）に合わせ、`product_locale == 'jp'`, `small_version == 1`, `split == 'test'` を評価する。主評価は各queryに付属する判定済み候補を並べ替える **candidate reranking** で、公式と同じgainによるnDCGを使う。

参照: [Amazon ESCI README](https://github.com/amazon-science/esci-data#task-1---query-product-ranking)、[公式QREL/Terrier変換コード](https://github.com/amazon-science/esci-data/blob/main/ranking/prepare_trec_eval_files.py)、[NIST trec_eval](https://github.com/usnistgov/trec_eval)。

## 評価規則

- 公式コードはQRELを `E=4, C=3, S=2, I=1` とし、Terrierへ `1=0, 2=0.01, 3=0.1, 4=1` を指定する。したがって実gainは **E=1, C=0.1, S=0.01, I=0**。DCGは `gain / log2(rank+1)`、主指標は全候補深さのmean nDCG。
- Task 1は全商品検索ではなく、queryごとに与えられた候補集合を全件採点して並べ替える問題。したがって主評価に未判定商品は存在しない。BM25で文字列matchしない候補もscore=0として候補末尾へ残す。全候補がゼロscoreならproduct_id順の決定的tie-breakを使い、別途zero-match率を必ず報告する。
- 参考として全商品からtop 100を検索するcorpus retrieval評価も出す。TREC慣行どおり未判定商品はgain 0として計算するが、ESCI qrelsはこの全商品検索用にpoolされたものではないため、絶対値は保守的な診断値であり公式Task 1スコアとは混ぜない。
- 結果が0件のqueryも除外せず、指標0として全query平均に含める。NIST `trec_eval -c` と同じcomplete-query averagingの考え方である。

In [ ]:
from pathlib import Path
import json, math, time
import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

BASE_URL = "http://localhost:9200"
INDEX_NAME = "amazon-jp"
DATA_DIR = Path("esci-data/shopping_queries_dataset")
TIMEOUT = 180
session = requests.Session()
root = session.get(BASE_URL, timeout=TIMEOUT)
root.raise_for_status()
count = session.get(f"{BASE_URL}/{INDEX_NAME}/_count", timeout=TIMEOUT)
count.raise_for_status()
print(
    f"OpenSearch {root.json()['version']['number']} / {INDEX_NAME}: {count.json()['count']:,} documents"
)

OpenSearch 3.7.0 / amazon-jp: 339,059 documents


## 1. 公式Task 1の日本語test split

In [ ]:
examples = pd.read_parquet(
    DATA_DIR / "shopping_queries_dataset_examples.parquet",
    filters=[
        ("product_locale", "=", "jp"),
        ("split", "=", "test"),
        ("small_version", "=", 1),
    ],
)
task1 = examples.sort_values(["query_id", "example_id"]).reset_index(drop=True)
query_stats = (
    task1.groupby("query_id")
    .agg(
        query=("query", "first"),
        candidates=("product_id", "size"),
        exact=("esci_label", lambda s: (s == "E").sum()),
        substitute=("esci_label", lambda s: (s == "S").sum()),
        complement=("esci_label", lambda s: (s == "C").sum()),
        irrelevant=("esci_label", lambda s: (s == "I").sum()),
    )
    .reset_index()
)
summary = pd.DataFrame(
    [
        {
            "queries": task1.query_id.nunique(),
            "judgements": len(task1),
            "mean candidates/query": len(task1) / task1.query_id.nunique(),
            "min candidates": query_stats.candidates.min(),
            "max candidates": query_stats.candidates.max(),
        }
    ]
)
display(summary.round(3))
display(task1.esci_label.value_counts().rename("judgements").to_frame())
assert count.json()["count"] == 339059
assert task1.query_id.nunique() == 3123 and len(task1) == 88789

,queries,judgements,mean candidates/query,min candidates,max candidates
0,3123,88789,28.431,8,86


,judgements
esci_label,
E,41597
S,29555
I,13604
C,4033


## 2. 公式gainとnDCG実装

In [ ]:
OFFICIAL_GAIN = {"E": 1.0, "C": 0.1, "S": 0.01, "I": 0.0}


def dcg(labels, k=None):
    if k is not None:
        labels = labels[:k]
    return sum(
        OFFICIAL_GAIN[label] / math.log2(rank + 2) for rank, label in enumerate(labels)
    )


def ndcg(run_ids, qrels, k=None):
    labels = [qrels.get(pid, "I") for pid in run_ids]
    ideal_labels = sorted(qrels.values(), key=lambda x: OFFICIAL_GAIN[x], reverse=True)
    denom = dcg(ideal_labels, k)
    return dcg(labels, k) / denom if denom else 0.0


toy = {"e": "E", "c": "C", "s": "S", "i": "I"}
assert abs(ndcg(["e", "c", "s", "i"], toy) - 1.0) < 1e-12
assert ndcg([], toy) == 0.0
display(
    pd.DataFrame(
        {
            "ESCI label": list(OFFICIAL_GAIN),
            "official gain": list(OFFICIAL_GAIN.values()),
        }
    )
)

,ESCI label,official gain
0,E,1.00
1,C,0.10
2,S,0.01
3,I,0.00


## 3. OpenSearchによる一括採点

In [ ]:
TITLE_FIELDS = ["product_title"]
MULTI_FIELDS = [
    "product_title^4",
    "product_brand^2",
    "product_bullet_point",
    "product_description^0.5",
]


def text_query(query, fields):
    if fields == TITLE_FIELDS:
        return {"match": {"product_title": {"query": query, "operator": "or"}}}
    return {
        "multi_match": {
            "query": query,
            "fields": fields,
            "type": "best_fields",
            "operator": "or",
        }
    }


def candidate_body(query, ids, fields):
    return {
        "size": len(ids),
        "_source": False,
        "query": {
            "bool": {
                "must": [text_query(query, fields)],
                "filter": [{"terms": {"product_id": ids}}],
            }
        },
    }


def corpus_body(query, fields=MULTI_FIELDS, size=100):
    return {"size": size, "_source": False, "query": text_query(query, fields)}


def run_msearch(specs, query_batch=64):
    outputs = {}
    step = query_batch * 3
    for start in range(0, len(specs), step):
        batch = specs[start : start + step]
        lines = []
        for key, body in batch:
            lines.extend(
                [
                    json.dumps({"index": INDEX_NAME}),
                    json.dumps(body, ensure_ascii=False),
                ]
            )
        r = session.post(
            f"{BASE_URL}/_msearch",
            headers={"Content-Type": "application/x-ndjson"},
            data=("\n".join(lines) + "\n").encode("utf-8"),
            timeout=TIMEOUT,
        )
        r.raise_for_status()
        responses = r.json()["responses"]
        if len(responses) != len(batch):
            raise RuntimeError("msearch response count mismatch")
        for (key, _), response in zip(batch, responses):
            if "error" in response:
                raise RuntimeError(f'{key}: {response["error"]}')
            outputs[key] = response["hits"]["hits"]
        if start == 0 or start + len(batch) == len(specs) or start % (step * 10) == 0:
            print(f"processed searches: {start + len(batch):,}/{len(specs):,}")
    return outputs


groups = {qid: g for qid, g in task1.groupby("query_id", sort=True)}
specs = []
for qid, g in groups.items():
    query = g["query"].iloc[0]
    ids = g.product_id.tolist()
    specs.append(((qid, "candidate_title"), candidate_body(query, ids, TITLE_FIELDS)))
    specs.append(((qid, "candidate_multi"), candidate_body(query, ids, MULTI_FIELDS)))
    specs.append(((qid, "corpus_multi"), corpus_body(query)))

started = time.time()
search_outputs = run_msearch(specs)
print(f"completed {len(specs):,} searches in {time.time() - started:.1f}s")

processed searches: 192/9,369


processed searches: 2,112/9,369


processed searches: 4,032/9,369


processed searches: 5,952/9,369


processed searches: 7,872/9,369


processed searches: 9,369/9,369
completed 9,369 searches in 3.4s


## 4. 公式candidate reranking評価

In [ ]:
candidate_rows = []
for qid, g in groups.items():
    qrels = dict(zip(g.product_id, g.esci_label))
    candidate_ids = list(qrels)
    for variant in ["candidate_title", "candidate_multi"]:
        hits = search_outputs[(qid, variant)]
        scores = {h["_id"]: h["_score"] for h in hits}
        ranked = sorted(
            candidate_ids,
            key=lambda pid: (pid not in scores, -scores.get(pid, 0.0), pid),
        )
        candidate_rows.append(
            {
                "query_id": qid,
                "query": g["query"].iloc[0],
                "variant": variant,
                "official_nDCG": ndcg(ranked, qrels),
                "official_nDCG_zero_match_0": (
                    0.0 if len(scores) == 0 else ndcg(ranked, qrels)
                ),
                "nDCG@10": ndcg(ranked, qrels, 10),
                "matched_candidates": len(scores),
                "candidate_count": len(candidate_ids),
                "candidate_match_rate": len(scores) / len(candidate_ids),
                "zero_match": len(scores) == 0,
            }
        )
candidate_eval = pd.DataFrame(candidate_rows)
candidate_summary = (
    candidate_eval.groupby("variant")
    .agg(
        official_nDCG=("official_nDCG", "mean"),
        nDCG_zero_match_0=("official_nDCG_zero_match_0", "mean"),
        nDCG_at_10=("nDCG@10", "mean"),
        mean_candidate_match_rate=("candidate_match_rate", "mean"),
        zero_match_queries=("zero_match", "sum"),
    )
    .reset_index()
)
display(candidate_summary.round(4))
display(
    Markdown(
        "**主結果:** `official_nDCG` は全候補を提出するAmazon ESCI Task 1形式。`nDCG_zero_match_0` は全候補が非matchのqueryを0にした保守値。"
    )
)

,variant,official_nDCG,nDCG_zero_match_0,nDCG_at_10,mean_candidate_match_rate,zero_match_queries
0,candidate_multi,0.8372,0.8061,0.6913,0.7633,129
1,candidate_title,0.8356,0.7957,0.6884,0.6701,167


**主結果:** `official_nDCG` は全候補を提出するAmazon ESCI Task 1形式。`nDCG_zero_match_0` は全候補が非matchのqueryを0にした保守値。

In [ ]:
worst = (
    candidate_eval[candidate_eval.variant == "candidate_multi"]
    .sort_values(["official_nDCG", "candidate_match_rate"])[
        [
            "query_id",
            "query",
            "official_nDCG",
            "nDCG@10",
            "candidate_match_rate",
            "zero_match",
        ]
    ]
    .head(20)
)
zero_examples = candidate_eval[
    (candidate_eval.variant == "candidate_multi") & candidate_eval.zero_match
][["query_id", "query", "candidate_count"]].head(20)
display(Markdown("### multi-fieldの下位20 query"))
display(worst.round(4))
display(Markdown("### zero-match query例"))
display(zero_examples)

### multi-fieldの下位20 query

,query_id,query,official_nDCG,nDCG@10,candidate_match_rate,zero_match
4377,125565,倉木麻衣 負けないで,0.1906,0.0000,0.9000,False
1279,116082,あえて結婚しない男子,0.2040,0.0098,0.4750,False
4861,126874,彼とデートしないでください,0.2414,0.0098,0.8750,False
5509,128639,神様には負けられない,0.2443,0.0039,0.6944,False
4501,125866,午後の紅茶 ストレート 無糖,0.2587,0.0030,1.0000,False
3989,124468,ラッシュガード メンズ アイス,0.2642,0.0274,0.6250,False
2213,118644,カリカリ,0.2715,0.0198,0.1875,False
727,59860,lanハブ2ポート,0.2770,0.0212,0.9792,False
255,11657,b08lv4lmwh ウェブカメラ マイクなし,0.2789,0.0439,1.0000,False
2639,120110,ジャグ,0.2837,0.0496,0.0000,True


### zero-match query例

,query_id,query,candidate_count
101,4492,4ｋビデオカード,9
237,10623,asca,16
305,20175,bskbw100sbk,34
347,28396,computer sticker,16
411,35486,drezea,40
475,40065,ff14,19
529,45045,gffmc,32
685,56387,jsports,16
757,62847,liquid_hack anti fog,40
823,72598,ncase m1,40


## 5. 全商品検索の参考診断（公式スコアではない）

ESCIの判定候補以外を取得した場合、その商品は未判定である。標準的なTREC nDCGでは未判定をgain 0として扱う。ただしESCI Task 1のqrelsは全商品検索runをpoolして作られたものではないため、ここでは `judged@10` を併記し、モデル選定の主指標にはしない。

In [ ]:
corpus_rows = []
for qid, g in groups.items():
    qrels = dict(zip(g.product_id, g.esci_label))
    run_ids = [h["_id"] for h in search_outputs[(qid, "corpus_multi")]]
    judged = set(qrels)
    positive = {pid for pid, label in qrels.items() if OFFICIAL_GAIN[label] > 0}
    corpus_rows.append(
        {
            "query_id": qid,
            "query": g["query"].iloc[0],
            "nDCG@10_unjudged0": ndcg(run_ids[:10], qrels, 10),
            "judged@10": len(set(run_ids[:10]) & judged) / 10,
            "Recall@100_judged_positive": len(set(run_ids) & positive) / len(positive),
            "hits": len(run_ids),
            "zero_hit": len(run_ids) == 0,
        }
    )
corpus_eval = pd.DataFrame(corpus_rows)
corpus_summary = pd.DataFrame(
    [
        {
            "queries": len(corpus_eval),
            "nDCG@10 (unjudged=0)": corpus_eval["nDCG@10_unjudged0"].mean(),
            "judged@10": corpus_eval["judged@10"].mean(),
            "Recall@100 (judged positive)": corpus_eval[
                "Recall@100_judged_positive"
            ].mean(),
            "zero-hit queries": int(corpus_eval.zero_hit.sum()),
        }
    ]
)
display(corpus_summary.round(4))

,queries,nDCG@10 (unjudged=0),judged@10,Recall@100 (judged positive),zero-hit queries
0,3123,0.3528,0.3734,0.4041,89


## 実行結果まとめ（2026-06-20）

|評価|title-only|multi-field|
|---|---:|---:|
|公式Task 1 candidate nDCG|0.8356|**0.8372**|
|zero-match queryを0にした保守nDCG|0.7957|**0.8061**|
|candidate nDCG@10（診断用）|0.6884|**0.6913**|
|候補match率|0.6701|**0.7633**|
|zero-match query数|167|**129**|

multi-field化は公式nDCGを+0.0016、保守nDCGを+0.0104改善し、zero-matchを38 query減らした。主な効果は順位の大幅改善より候補coverageの改善。

全商品検索の参考値は nDCG@10=0.3528、judged@10=0.3734、judged positive Recall@100=0.4041、zero-hit=89。上位10件の約63%が未判定なので、このnDCGを公式Task 1スコアやモデルの最終判定に使わない。

## 解釈上の注意

1. 公式比較に使うのはSection 4のcandidate reranking `official_nDCG`。Amazon公式baselineの0.83は全locale統合かつ学習済みneural rerankerの値で、今回の日本語BM25だけの値と条件が完全には同一でない。
2. `C` は `S` より高gain（0.1対0.01）である点が通常のE>S>C>Iという直感と異なるが、公式変換コードをそのまま踏襲した。
3. candidate内の非matchを捨てるとTask 1の候補全件rankingにならないため、score=0で末尾に残した。zero-match query数はtie-break依存のnDCGを見抜くために別報告した。
4. corpus retrievalでは未判定をIと意味的に断定していない。計算上gain 0としただけであり、judged率が低い場合は追加pooling・人手判定、またはbpref等の検討が必要。
5. 全queryを平均対象にし、zero-hitを除外しない。除外すると検索不能queryを隠して指標が上振れする。